In [ ]:
# ============================================
# CELL 1 — Import Libraries & Load Dataset
# ============================================
import pandas as pd
import numpy as np
import sqlite3
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(message)s')

# Load main dataset
df = pd.read_csv('data/clean/car_prices.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Sample data:")
df.head(3)

In [ ]:
# ============================================
# CELL 2 — Data Quality & Validation Report
# ============================================

print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

# Missing values
print("\nMissing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')
print(quality_report)

# Data types
print("\nData Types:")
print(df.dtypes)

# Key stats
print("\nSelling Price Stats:")
print(df['sellingprice'].describe().round(2))

print("\nTop 5 Vehicle Makes:")
print(df['make'].value_counts().head())

print("\nYear Range:")
print(f"Oldest: {df['year'].min()} | Newest: {df['year'].max()}")

print("\nValidation Complete!")

In [ ]:
# ============================================
# CELL 3 — Data Cleaning Pipeline
# ============================================

initial_count = len(df)
print(f"Starting pipeline with {initial_count:,} records...")

# Step 1 — Drop rows missing critical fields
df_clean = df.dropna(subset=['make', 'model', 'sellingprice', 'year', 'odometer'])
print(f"After dropping critical nulls: {len(df_clean):,} rows")

# Step 2 — Remove invalid prices
df_clean = df_clean[df_clean['sellingprice'] > 500]
df_clean = df_clean[df_clean['sellingprice'] < 200000]
print(f"After price filter ($500-$200k): {len(df_clean):,} rows")

# Step 3 — Remove invalid odometer
df_clean = df_clean[df_clean['odometer'] > 0]
df_clean = df_clean[df_clean['odometer'] < 500000]
print(f"After odometer filter: {len(df_clean):,} rows")

# Step 4 — Fill missing values
df_clean['transmission'] = df_clean['transmission'].fillna('unknown')
df_clean['body'] = df_clean['body'].fillna('unknown')
print(f"Missing values filled")

# Step 5 — Standardise text columns
df_clean['make'] = df_clean['make'].str.strip().str.title()
df_clean['model'] = df_clean['model'].str.strip().str.title()
df_clean['body'] = df_clean['body'].str.strip().str.title()
print(f"Text columns standardised")

# Step 6 — Add derived columns
df_clean['vehicle_age'] = 2026 - df_clean['year']
df_clean['price_vs_market'] = (df_clean['sellingprice'] - df_clean['mmr']).round(2)
df_clean['price_vs_market_pct'] = ((df_clean['price_vs_market'] / df_clean['mmr']) * 100).round(2)
print(f"Derived columns added")

# Final summary
removed = initial_count - len(df_clean)
removal_pct = round(removed / initial_count * 100, 2)

print(f"\n{'='*50}")
print(f"PIPELINE COMPLETE")
print(f"{'='*50}")
print(f"Records in:      {initial_count:,}")
print(f"Records out:     {len(df_clean):,}")
print(f"Records removed: {removed:,} ({removal_pct}%)")
print(f"Data quality:    {100 - removal_pct}% retained")
print(f"\nNew columns added:")
print(f"  → vehicle_age")
print(f"  → price_vs_market")
print(f"  → price_vs_market_pct")

In [ ]:
# ============================================
# CELL 4 — Load Clean Data into SQLite Database
# ============================================
import sqlite3
import os

# Create database folder
os.makedirs('database', exist_ok=True)

# Connect to SQLite
conn = sqlite3.connect('database/automotive_rd.db')
cursor = conn.cursor()

# Save clean dataframe to SQL table
df_clean.to_sql('vehicle_sales', conn, if_exists='replace', index=False)

# Verify it loaded correctly
result = pd.read_sql_query("SELECT COUNT(*) as total_records FROM vehicle_sales", conn)
print(f"Database created successfully!")
print(f"Records in database: {result['total_records'][0]:,}")

# Preview with SQL query
print(f"\nFirst SQL Query — Top 5 Makes by Volume:")
query = """
    SELECT 
        make,
        COUNT(*) as total_sold,
        ROUND(AVG(sellingprice), 2) as avg_price
    FROM vehicle_sales
    GROUP BY make
    ORDER BY total_sold DESC
    LIMIT 5
"""
top_makes = pd.read_sql_query(query, conn)
print(top_makes.to_string(index=False))

print(f"\nSQLite database ready at: database/automotive_rd.db")

In [ ]:
# ============================================
# CELL 5 — SQL KPI Analysis
# ============================================

print("=" * 55)
print("AUTOMOTIVE MARKET KPI DASHBOARD")
print("=" * 55)

# KPI 1 — Market share by body type
print("\n KPI 1 — Market Share by Vehicle Segment:")
kpi1 = pd.read_sql_query("""
    SELECT 
        body,
        COUNT(*) as units_sold,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM vehicle_sales), 2) as market_share_pct,
        ROUND(AVG(sellingprice), 0) as avg_price,
        ROUND(AVG(vehicle_age), 1) as avg_age_years
    FROM vehicle_sales
    WHERE body != 'Unknown'
    GROUP BY body
    ORDER BY units_sold DESC
    LIMIT 8
""", conn)
print(kpi1.to_string(index=False))

# KPI 2 — Price depreciation by age
print("\n KPI 2 — Price Depreciation by Vehicle Age:")
kpi2 = pd.read_sql_query("""
    SELECT 
        vehicle_age,
        COUNT(*) as sample_size,
        ROUND(AVG(sellingprice), 0) as avg_resale_price,
        ROUND(AVG(odometer), 0) as avg_mileage
    FROM vehicle_sales
    WHERE vehicle_age BETWEEN 1 AND 15
    GROUP BY vehicle_age
    ORDER BY vehicle_age
""", conn)
print(kpi2.to_string(index=False))

# KPI 3 — Brand premium index
print("\nKPI 3 — Brand Premium Index (vs Market Value):")
kpi3 = pd.read_sql_query("""
    SELECT 
        make,
        COUNT(*) as total_sold,
        ROUND(AVG(sellingprice), 0) as avg_price,
        ROUND(AVG(price_vs_market), 0) as avg_vs_mmr,
        ROUND(AVG(price_vs_market_pct), 2) as premium_pct
    FROM vehicle_sales
    WHERE mmr > 0
    GROUP BY make
    HAVING COUNT(*) > 1000
    ORDER BY premium_pct DESC
    LIMIT 10
""", conn)
print(kpi3.to_string(index=False))

# KPI 4 — Condition impact on price
print("\nKPI 4 — Vehicle Condition Impact on Price:")
kpi4 = pd.read_sql_query("""
    SELECT 
        ROUND(condition, 0) as condition_score,
        COUNT(*) as vehicles,
        ROUND(AVG(sellingprice), 0) as avg_price,
        ROUND(AVG(odometer), 0) as avg_mileage
    FROM vehicle_sales
    WHERE condition IS NOT NULL
    GROUP BY ROUND(condition, 0)
    ORDER BY condition_score DESC
""", conn)
print(kpi4.to_string(index=False))

print("\nAll KPIs calculated successfully!")

In [ ]:
# ============================================
# CELL 6 — Data Visualisations
# ============================================
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('outputs/charts', exist_ok=True)
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Automotive R&D Intelligence — Market Overview', 
             fontsize=16, fontweight='bold', color='white', y=1.02)

# Chart 1 — Market Share by Segment
top_segments = kpi1.head(6)
colors = ['#00D4FF', '#0099CC', '#006699', '#004466', '#002233', '#001122']
axes[0,0].barh(top_segments['body'], top_segments['market_share_pct'], color=colors)
axes[0,0].set_title('Market Share by Vehicle Segment (%)', color='white', fontweight='bold')
axes[0,0].set_xlabel('Market Share %', color='white')
for i, v in enumerate(top_segments['market_share_pct']):
    axes[0,0].text(v + 0.2, i, f'{v}%', color='white', va='center', fontweight='bold')

# Chart 2 — Price Depreciation Curve
axes[0,1].plot(kpi2['vehicle_age'], kpi2['avg_resale_price'], 
               marker='o', color='#00FF88', linewidth=2.5, markersize=8)
axes[0,1].fill_between(kpi2['vehicle_age'], kpi2['avg_resale_price'], 
                        alpha=0.2, color='#00FF88')
axes[0,1].set_title('Price Depreciation by Vehicle Age', color='white', fontweight='bold')
axes[0,1].set_xlabel('Vehicle Age (Years)', color='white')
axes[0,1].set_ylabel('Avg Resale Price ($)', color='white')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Chart 3 — Brand Premium Index
axes[1,0].barh(kpi3['make'], kpi3['premium_pct'], 
               color=['#00FF88' if x > 0 else '#FF4444' for x in kpi3['premium_pct']])
axes[1,0].set_title('Brand Premium Index (% vs Market Value)', color='white', fontweight='bold')
axes[1,0].set_xlabel('Premium %', color='white')
axes[1,0].axvline(x=0, color='white', linestyle='--', alpha=0.5)

# Chart 4 — Condition vs Price
condition_summary = kpi4[kpi4['condition_score'].between(20, 50)]
axes[1,1].scatter(condition_summary['condition_score'], 
                  condition_summary['avg_price'],
                  color='#FF9900', alpha=0.8, s=80)
axes[1,1].set_title('Condition Score vs Average Price', color='white', fontweight='bold')
axes[1,1].set_xlabel('Condition Score', color='white')
axes[1,1].set_ylabel('Average Price ($)', color='white')
axes[1,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Style all axes
for ax in axes.flat:
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#444444')
    ax.spines['left'].set_color('#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/charts/market_overview.png', dpi=300, bbox_inches='tight',
            facecolor='#1a1a1a')
plt.show()
print("Charts saved to outputs/charts/market_overview.png")

In [ ]:
# ============================================
# CELL 7 — Machine Learning: Price Prediction
# ============================================
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

print("🔧 Preparing features for ML model...")

# Encode categorical columns
le_make = LabelEncoder()
le_body = LabelEncoder()
le_transmission = LabelEncoder()

df_ml = df_clean.copy()
df_ml['make_encoded'] = le_make.fit_transform(df_ml['make'].astype(str))
df_ml['body_encoded'] = le_body.fit_transform(df_ml['body'].astype(str))
df_ml['transmission_encoded'] = le_transmission.fit_transform(df_ml['transmission'].astype(str))

# Select features
features = [
    'year', 
    'odometer', 
    'condition', 
    'mmr', 
    'vehicle_age',
    'make_encoded', 
    'body_encoded',
    'transmission_encoded'
]

df_ml = df_ml.dropna(subset=features + ['sellingprice'])
X = df_ml[features]
y = df_ml['sellingprice']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f" Training set: {len(X_train):,} records")
print(f" Test set:     {len(X_test):,} records")
print(f"\n🤖 Training models — please wait...\n")

# Train 3 models
results = {}

# Model 1 — Linear Regression (baseline)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
results['Linear Regression'] = {
    'R2': round(r2_score(y_test, lr_preds), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, lr_preds)), 2)
}
print(f" Linear Regression done")

# Model 2 — Random Forest
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
results['Random Forest'] = {
    'R2': round(r2_score(y_test, rf_preds), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, rf_preds)), 2)
}
print(f" Random Forest done")

# Model 3 — Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
results['Gradient Boosting'] = {
    'R2': round(r2_score(y_test, gb_preds), 3),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, gb_preds)), 2)
}
print(f" Gradient Boosting done")

# Print results
print(f"\n{'='*55}")
print(f"🏆 MODEL COMPARISON RESULTS")
print(f"{'='*55}")
print(f"{'Model':<25} {'R² Score':>10} {'RMSE ($)':>12}")
print(f"{'-'*55}")
for name, metrics in results.items():
    print(f"{name:<25} {metrics['R2']:>10} {metrics['RMSE']:>12,.2f}")

print(f"\n💡 R² Score: closer to 1.0 = better accuracy")
print(f"💡 RMSE: average prediction error in dollars")

In [ ]:
# ============================================
# CELL 8 — Fixed Actual vs Predicted Chart
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ML Model — Price Prediction Analysis', 
             fontsize=15, fontweight='bold', color='white')

# Chart 1 — Feature Importance (working fine)
feature_names = ['Year', 'Odometer', 'Condition', 
                 'MMR Value', 'Vehicle Age', 
                 'Make', 'Body Type', 'Transmission']
importances = gb.feature_importances_
sorted_idx = np.argsort(importances)

axes[0].barh([feature_names[i] for i in sorted_idx], 
              importances[sorted_idx], color='#00D4FF')
axes[0].set_title('Feature Importance — What Drives Price?', 
                   color='white', fontweight='bold')
axes[0].set_xlabel('Importance Score', color='white')
axes[0].tick_params(colors='white')
for i, (idx, imp) in enumerate(zip(sorted_idx, importances[sorted_idx])):
    axes[0].text(imp + 0.001, i, f'{imp*100:.1f}%', 
                 va='center', color='white', fontsize=9)

# Chart 2 — Fixed Actual vs Predicted
y_test_array = np.array(y_test)
sample_size = 2000
sample_idx = np.random.choice(len(y_test_array), sample_size, replace=False)
actual_sample = y_test_array[sample_idx]
predicted_sample = gb_preds[sample_idx]

# Get price range for axis limits
max_price = min(actual_sample.max(), 80000)
min_price = actual_sample.min()

axes[1].scatter(actual_sample, predicted_sample, 
                alpha=0.4, color='#FF9900', s=20)
axes[1].plot([min_price, max_price], 
             [min_price, max_price],
             'white', linestyle='--', linewidth=2, 
             label=f'Perfect Prediction Line')
axes[1].set_title('Actual vs Predicted Price (sample 2,000)', 
                   color='white', fontweight='bold')
axes[1].set_xlabel('Actual Price ($)', color='white')
axes[1].set_ylabel('Predicted Price ($)', color='white')
axes[1].xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
axes[1].legend(loc='upper left', labelcolor='white')
axes[1].tick_params(colors='white')

for ax in axes:
    ax.spines['bottom'].set_color('#444444')
    ax.spines['left'].set_color('#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/charts/ml_model.png', 
            dpi=300, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

# Feature importance breakdown
print(f"\n FEATURE IMPORTANCE BREAKDOWN:")
print(f"{'='*45}")
for idx in sorted_idx[::-1]:
    bar = '█' * int(importances[idx] * 50)
    print(f"{feature_names[idx]:<15} {bar} {importances[idx]*100:.1f}%")

print(f"\n KEY FINDING: MMR Market Value dominates price prediction")
print(f"   This confirms market benchmarking is the #1 pricing signal")

In [ ]:
# ============================================
# CELL 9 — EV Transition & Sustainability Analysis
# ============================================

print("Running EV Transition & Sustainability Analysis...")
print("=" * 55)

# Define EV, Hybrid and ICE brands
ev_brands = ['Tesla', 'Rivian', 'Lucid', 'Polestar', 'Fisker']
hybrid_brands = ['Toyota', 'Honda', 'Lexus', 'Ford', 'Hyundai']
ice_brands = ['Chevrolet', 'Dodge', 'Chrysler', 'Ram', 'Jeep']

# Classify powertrain
def classify_powertrain(make):
    if make in ev_brands:
        return 'EV'
    elif make in hybrid_brands:
        return 'Hybrid/ICE'
    elif make in ice_brands:
        return 'ICE'
    else:
        return 'ICE'

df_clean['powertrain'] = df_clean['make'].apply(classify_powertrain)

# ── ANALYSIS 1 — Powertrain market share
print("\n Powertrain Market Share:")
powertrain_share = df_clean.groupby('powertrain').agg(
    total_vehicles=('make', 'count'),
    avg_price=('sellingprice', 'mean'),
    avg_age=('vehicle_age', 'mean')
).round(2)
powertrain_share['market_share_pct'] = (
    powertrain_share['total_vehicles'] / 
    len(df_clean) * 100).round(2)
print(powertrain_share.to_string())

# ── ANALYSIS 2 — CO₂ emissions estimation
print("\n Fleet Carbon Footprint Analysis:")

emissions_factors = {
    'ICE':        {'sedan': 145, 'suv': 210, 'truck': 240, 'default': 175},
    'Hybrid/ICE': {'sedan':  95, 'suv': 140, 'truck': 160, 'default': 115},
    'EV':         {'sedan':   0, 'suv':   0, 'truck':   0, 'default':   0}
}

ANNUAL_KM = 11900  # UK average annual mileage
CARBON_PRICE = 45  # UK ETS £/tonne

def get_co2(row):
    pt = row['powertrain']
    body = str(row['body']).lower()
    if 'suv' in body:
        segment = 'suv'
    elif 'truck' in body or 'cab' in body:
        segment = 'truck'
    elif 'sedan' in body:
        segment = 'sedan'
    else:
        segment = 'default'
    return emissions_factors[pt][segment]

df_clean['co2_per_km'] = df_clean.apply(get_co2, axis=1)
df_clean['annual_co2_kg'] = (df_clean['co2_per_km'] * ANNUAL_KM / 1000).round(2)
df_clean['annual_carbon_cost_gbp'] = (
    df_clean['annual_co2_kg'] / 1000 * CARBON_PRICE).round(2)

carbon_summary = df_clean.groupby('powertrain').agg(
    vehicles=('make', 'count'),
    avg_co2_kg_year=('annual_co2_kg', 'mean'),
    total_co2_tonnes=('annual_co2_kg', lambda x: round(x.sum()/1000, 0)),
    avg_carbon_cost_gbp=('annual_carbon_cost_gbp', 'mean')
).round(2)
print(carbon_summary.to_string())

# ── ANALYSIS 3 — ZEV Mandate Compliance
print("\n UK ZEV Mandate Targets vs Dataset Trends:")
zev_targets = {
    2024: 22, 2025: 28, 2026: 33,
    2027: 38, 2028: 52, 2029: 66,
    2030: 80, 2035: 100
}

ev_count = len(df_clean[df_clean['powertrain'] == 'EV'])
total_count = len(df_clean)
current_ev_pct = round(ev_count / total_count * 100, 2)

print(f"\n  Current EV share in dataset:  {current_ev_pct}%")
print(f"  UK ZEV target for 2026:       33%")
print(f"  Compliance gap:               {round(33 - current_ev_pct, 2)}%")
print(f"\n  UK ZEV Mandate Roadmap:")
for year, target in zev_targets.items():
    bar = '█' * (target // 5)
    print(f"  {year}: {bar} {target}%")

print(f"\n EV & Sustainability Analysis Complete!")

In [ ]:
# ============================================
# CELL 10 — Sustainability & EV Charts
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(' EV Transition & Sustainability Intelligence',
             fontsize=16, fontweight='bold', color='white', y=1.02)

# Chart 1 — Powertrain Market Share (Donut)
powertrain_counts = df_clean['powertrain'].value_counts()
colors_pt = ['#00FF88', '#00D4FF', '#FF4444']
wedges, texts, autotexts = axes[0,0].pie(
    powertrain_counts.values,
    labels=powertrain_counts.index,
    autopct='%1.1f%%',
    colors=colors_pt,
    startangle=90,
    wedgeprops=dict(width=0.6)
)
for text in texts + autotexts:
    text.set_color('white')
    text.set_fontsize(11)
axes[0,0].set_title('Fleet Powertrain Distribution', 
                     color='white', fontweight='bold', pad=20)

# Chart 2 — CO₂ Emissions by Powertrain
powertrains = ['EV', 'Hybrid/ICE', 'ICE']
co2_values = [0, 1378, 2085]
bar_colors = ['#00FF88', '#FFD700', '#FF4444']
bars = axes[0,1].bar(powertrains, co2_values, color=bar_colors, width=0.5)
axes[0,1].set_title('Average Annual CO₂ per Vehicle (kg)',
                     color='white', fontweight='bold')
axes[0,1].set_ylabel('CO₂ kg/year', color='white')
axes[0,1].tick_params(colors='white')
for bar, val in zip(bars, co2_values):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, 
                   bar.get_height() + 20,
                   f'{val:,} kg', ha='center', 
                   color='white', fontweight='bold')
axes[0,1].set_facecolor('#1a1a1a')

# Chart 3 — ZEV Mandate Compliance Gap
years = list(zev_targets.keys())
targets = list(zev_targets.values())
actual_2026 = current_ev_pct

axes[1,0].plot(years, targets, marker='D', color='#00FF88', 
               linewidth=2.5, markersize=8, label='UK ZEV Target')
axes[1,0].axhline(y=actual_2026, color='#FF4444', 
                   linestyle='--', linewidth=2,
                   label=f'Current EV Share ({actual_2026}%)')
axes[1,0].fill_between(years, actual_2026, targets,
                        alpha=0.2, color='#FF4444',
                        label='Compliance Gap')
axes[1,0].set_title('UK ZEV Mandate — Compliance Gap',
                     color='white', fontweight='bold')
axes[1,0].set_xlabel('Year', color='white')
axes[1,0].set_ylabel('EV Market Share (%)', color='white')
axes[1,0].tick_params(colors='white')
axes[1,0].legend(labelcolor='white', facecolor='#2a2a2a')
axes[1,0].set_facecolor('#1a1a1a')

# Chart 4 — Carbon Cost by Segment
segment_carbon = df_clean.groupby('body').agg(
    avg_carbon_cost=('annual_carbon_cost_gbp', 'mean'),
    count=('make', 'count')
).query('count > 1000').sort_values('avg_carbon_cost', ascending=True)

colors_seg = ['#00FF88' if x < 60 else '#FFD700' 
              if x < 80 else '#FF4444' 
              for x in segment_carbon['avg_carbon_cost']]
axes[1,1].barh(segment_carbon.index, 
               segment_carbon['avg_carbon_cost'],
               color=colors_seg)
axes[1,1].set_title('Avg Annual Carbon Cost by Segment (£)',
                     color='white', fontweight='bold')
axes[1,1].set_xlabel('Carbon Cost (£/year)', color='white')
axes[1,1].tick_params(colors='white')
for i, val in enumerate(segment_carbon['avg_carbon_cost']):
    axes[1,1].text(val + 0.5, i, f'£{val:.0f}', 
                   va='center', color='white', fontsize=9)
axes[1,1].set_facecolor('#1a1a1a')

for ax in axes.flat:
    ax.spines['bottom'].set_color('#444444')
    ax.spines['left'].set_color('#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/charts/sustainability.png',
            dpi=300, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()
print("Sustainability charts saved!")

In [ ]:
# ============================================
# CELL 11 — Save All Outputs & Final Summary
# ============================================
import os

print("Saving all project outputs...")
print("=" * 55)

# Save cleaned dataset
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/vehicle_sales_clean.csv', index=False)
print(f"Clean dataset saved — {len(df_clean):,} records")

# Save KPI reports
os.makedirs('outputs/kpi_reports', exist_ok=True)

kpi1.to_csv('outputs/kpi_reports/market_share_by_segment.csv', index=False)
kpi2.to_csv('outputs/kpi_reports/price_depreciation.csv', index=False)
kpi3.to_csv('outputs/kpi_reports/brand_premium_index.csv', index=False)
kpi4.to_csv('outputs/kpi_reports/condition_impact.csv', index=False)
carbon_summary.to_csv('outputs/kpi_reports/carbon_sustainability.csv')
print(f"5 KPI reports saved")

# Save ML model
import pickle
os.makedirs('models', exist_ok=True)
with open('models/price_prediction_model.pkl', 'wb') as f:
    pickle.dump(gb, f)
print(f"ML model saved")

# Final project summary
print(f"\n{'='*55}")
print(f"🏆 PROJECT COMPLETE — FINAL SUMMARY")
print(f"{'='*55}")
print(f"""
DATA PIPELINE
   Records processed:     {len(df):,}
   Records retained:      {len(df_clean):,} (96.95%)
   Database:              SQLite ✅

SQL KPI ANALYSIS  
   Market segments:       {kpi1.shape[0]} analysed
   Price depreciation:    {kpi2.shape[0]} age groups
   Brand premium index:   {kpi3.shape[0]} brands ranked
   Condition impact:      {kpi4.shape[0]} score levels

MACHINE LEARNING
   Training records:      424,451
   Test records:          106,113
   Best model:            Gradient Boosting
   R² Accuracy:           97.5%
   Avg price error:       $1,527

SUSTAINABILITY
   Fleet CO₂ analysed:    {len(df_clean):,} vehicles
   ICE avg CO₂/year:      2,085 kg
   Hybrid avg CO₂/year:   1,378 kg  
   EV avg CO₂/year:       0 kg
   ZEV compliance gap:    32.99%

OUTPUT FILES
   Charts saved:          3 dashboards
   KPI reports:           5 CSV files
   ML model:              Saved ✅
   Clean data:            Saved ✅
""")
print("Portfolio project ready for GitHub!")

In [ ]:
# ============================================
# CELL 12 — Export Optimised Files for Tableau
# ============================================
import os
os.makedirs('outputs/tableau', exist_ok=True)

# File 1 — Main dashboard data (sampled for Tableau performance)
tableau_main = df_clean[[
    'year', 'make', 'model', 'body', 'transmission',
    'condition', 'odometer', 'sellingprice', 'mmr',
    'vehicle_age', 'price_vs_market', 'price_vs_market_pct',
    'state', 'color', 'powertrain', 'annual_co2_kg',
    'annual_carbon_cost_gbp'
]].sample(50000, random_state=42)  # 50k rows — perfect for Tableau

tableau_main.to_csv('outputs/tableau/main_dashboard.csv', index=False)
print(f" Main dashboard data: {len(tableau_main):,} records")

# File 2 — KPI summary
tableau_kpi = pd.DataFrame({
    'metric': [
        'Total Records', 'Avg Selling Price', 
        'Avg Vehicle Age', 'Data Quality %',
        'ML Model Accuracy', 'ICE CO2 kg/year',
        'Hybrid CO2 kg/year', 'EV CO2 kg/year'
    ],
    'value': [
        541798, 13611, 15.7, 96.95,
        97.5, 2085, 1378, 0
    ],
    'unit': [
        'vehicles', 'USD', 'years', '%',
        '%', 'kg', 'kg', 'kg'
    ]
})
tableau_kpi.to_csv('outputs/tableau/kpi_summary.csv', index=False)
print(f"KPI summary exported")

# File 3 — Brand analysis
tableau_brands = df_clean.groupby(['make', 'powertrain']).agg(
    total_sold=('sellingprice', 'count'),
    avg_price=('sellingprice', 'mean'),
    avg_condition=('condition', 'mean'),
    avg_mileage=('odometer', 'mean'),
    avg_co2=('annual_co2_kg', 'mean'),
    avg_carbon_cost=('annual_carbon_cost_gbp', 'mean')
).round(2).reset_index()
tableau_brands.to_csv('outputs/tableau/brand_analysis.csv', index=False)
print(f"Brand analysis: {len(tableau_brands)} brands")

# File 4 — Sustainability summary
tableau_sustainability = df_clean.groupby(['powertrain', 'body']).agg(
    vehicles=('make', 'count'),
    avg_co2_kg=('annual_co2_kg', 'mean'),
    total_co2_tonnes=('annual_co2_kg', lambda x: round(x.sum()/1000, 1)),
    avg_carbon_cost=('annual_carbon_cost_gbp', 'mean'),
    avg_price=('sellingprice', 'mean')
).round(2).reset_index()
tableau_sustainability.to_csv('outputs/tableau/sustainability.csv', index=False)
print(f"Sustainability data exported")

print(f"\n{'='*50}")
print(f" All Tableau files saved to outputs/tableau/")
print(f"\nFiles ready:")
for f in os.listdir('outputs/tableau/'):
    size = os.path.getsize(f'outputs/tableau/{f}')
    print(f"   {f} ({size/1024:.1f} KB)")
print(f"\n Ready for Tableau!")